In [ ]:
# | default_exp features.mds_exon

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class MDSExonEvaluator(FeatureEvaluator):
    """Exon-level MDS distributions with cross-exon statistics.

    Per-gene exon-level: mean and std of MDS across exons per gene.
    Cross-exon derived: global mean, std, skew, fraction diverged (|MDS|>2).
    """

    name = "MDSExon"
    source_file = ".MDS.exon.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            # Per-gene grouped extraction
            if "gene" in cols and "mds" in cols:
                gene_grouped = df.groupby("gene")["mds"]
                for gene, vals in gene_grouped:
                    if str(gene) == "NAN":
                        continue
                    g = str(gene).replace(" ", "_").replace("-", "_")
                    extracted[f"{g}_mds_exon_mean"] = float(vals.mean())
                    if len(vals) > 1:
                        extracted[f"{g}_mds_exon_std"] = float(vals.std())

            # --- Cross-exon distribution statistics ---
            if "mds" in cols:
                all_mds = df["mds"].dropna()
                if len(all_mds) > 1:
                    extracted["mds_exon_global_mean"] = float(all_mds.mean())
                    extracted["mds_exon_global_std"] = float(all_mds.std())
                    extracted["mds_exon_global_skew"] = float(all_mds.skew())
                    extracted["mds_exon_frac_diverged"] = float(
                        (all_mds.abs() > 2).mean()
                    )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = MDSExonEvaluator()
df = pd.DataFrame([{"gene": "TP53", "mds": 0.8}, {"gene": "TP53", "mds": 0.4}])
res = evaluator.extract(df)
assert "TP53_mds_exon_mean" in res
assert "TP53_mds_exon_std" in res